# Car Dekho Data Analysis

This notebook analyzes the Car Dekho used-vehicle dataset and answers the 25 questions provided in the assignment.

### Analysis covered
- Dataset size and missing values
- Manufacturing year range
- Minimum and maximum selling price
- Unique vehicle models
- Most frequently occurring vehicle
- Fuel type, seller type, transmission and ownership analysis
- Vehicle depreciation
- Factors affecting depreciation
- Two-wheeler analysis
- Car-only analysis
- Exceptional resale/depreciation cases


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

file_path = "1776311302-P3-Car Market Trends Analysis with Car Dekho Data.csv"
data = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", data.shape)
display(data.head())


## 1. Basic Dataset Information

In [ ]:
print("Number of records:", len(data))
print("Number of columns:", len(data.columns))
print("\nColumns:")
print(list(data.columns))

print("\nData types:")
display(data.dtypes.to_frame("Data Type"))


## 2. Missing Values

In [ ]:
missing = data.isnull().sum().to_frame("Missing Values")
missing["Missing Percentage"] = (missing["Missing Values"] / len(data) * 100).round(2)
display(missing)

if data.isnull().sum().sum() == 0:
    print("Conclusion: There are no missing values in the dataset.")
else:
    print("Conclusion: Missing values are present.")


## 3. Manufacturing Year Range

In [ ]:
print("Oldest manufacturing year:", data["Year"].min())
print("Newest manufacturing year:", data["Year"].max())
print(f"Vehicles are present from {data['Year'].min()} to {data['Year'].max()}.")


## 4. Lowest and Highest Selling Price

In [ ]:
min_row = data.loc[data["Selling_Price"].idxmin()]
max_row = data.loc[data["Selling_Price"].idxmax()]

print("Lowest selling price:", min_row["Selling_Price"], "lakh")
display(min_row.to_frame("Value"))

print("\nHighest selling price:", max_row["Selling_Price"], "lakh")
display(max_row.to_frame("Value"))


## 5. Number of Different Vehicles

In [ ]:
unique_vehicles = data["Car_Name"].nunique()
print("Number of unique vehicle models:", unique_vehicles)

print("\nMost frequently occurring vehicles:")
display(data["Car_Name"].value_counts().head(10).to_frame("Records"))


## 6. Fuel Type Analysis

In [ ]:
fuel_counts = data["Fuel_Type"].value_counts()
display(fuel_counts.to_frame("Records"))

print("CNG vehicles:", (data["Fuel_Type"] == "CNG").sum())

fuel_summary = data.groupby("Fuel_Type").agg(
    Records=("Car_Name", "size"),
    Average_Selling_Price=("Selling_Price", "mean")
).sort_values("Records", ascending=False)

display(fuel_summary.round(2))


## 7. Seller Type Analysis

In [ ]:
seller_counts = data["Seller_Type"].value_counts()
display(seller_counts.to_frame("Records"))

print("Vehicles sold by Individuals:",
      (data["Seller_Type"] == "Individual").sum())

seller_summary = data.groupby("Seller_Type").agg(
    Records=("Car_Name", "size"),
    Average_Selling_Price=("Selling_Price", "mean")
)

display(seller_summary.round(2))


## 8. Transmission Analysis

In [ ]:
transmission_counts = data["Transmission"].value_counts()
display(transmission_counts.to_frame("Records"))

print("Automatic vehicles:",
      (data["Transmission"] == "Automatic").sum())

transmission_summary = data.groupby("Transmission").agg(
    Records=("Car_Name", "size"),
    Average_Selling_Price=("Selling_Price", "mean")
)

display(transmission_summary.round(2))


## 9. Ownership Analysis

In [ ]:
owner_counts = data["Owner"].value_counts().sort_index()
display(owner_counts.to_frame("Records"))

print("First-owner vehicles (Owner = 0):",
      (data["Owner"] == 0).sum())


## 10. Depreciation Analysis

**Depreciation = Present Price − Selling Price**

**Depreciation % = (Present Price − Selling Price) / Present Price × 100**


In [ ]:
data["Depreciation"] = data["Present_Price"] - data["Selling_Price"]
data["Depreciation_Percent"] = (
    data["Depreciation"] / data["Present_Price"]
) * 100

most_depreciated = data.loc[data["Depreciation"].idxmax()]
least_depreciated = data.loc[data["Depreciation"].idxmin()]

print("Most depreciated vehicle:")
display(most_depreciated.to_frame("Value"))

print("\nLeast depreciated vehicle:")
display(least_depreciated.to_frame("Value"))


## 11. Brand-Level Depreciation

In [ ]:
# Extract brand as the first word of the vehicle name
data["Brand"] = data["Car_Name"].str.split().str[0]

brand_summary = data.groupby("Brand").agg(
    Records=("Car_Name", "size"),
    Average_Depreciation=("Depreciation", "mean"),
    Average_Depreciation_Percent=("Depreciation_Percent", "mean")
)

# Avoid unstable conclusions from brands with very few observations
brand_summary_5plus = brand_summary[brand_summary["Records"] >= 5].sort_values(
    "Average_Depreciation_Percent"
)

display(brand_summary_5plus.round(2))

print("Among brands with at least 5 records, lower average percentage depreciation indicates better value retention.")


## 12. Factors Affecting Depreciation

In [ ]:
data["Age"] = data["Year"].max() - data["Year"]

correlations = data[
    ["Depreciation_Percent", "Age", "Kms_Driven", "Present_Price", "Selling_Price"]
].corr()

display(correlations.round(3))

print("Correlation of age with depreciation percentage:",
      round(correlations.loc["Age", "Depreciation_Percent"], 3))

print("Correlation of kilometres driven with depreciation percentage:",
      round(correlations.loc["Kms_Driven", "Depreciation_Percent"], 3))

print("Correlation of kilometres driven with selling price:",
      round(correlations.loc["Kms_Driven", "Selling_Price"], 3))


## 13. Selling Price vs Vehicle Age and Kilometres

In [ ]:
print("Average selling price by vehicle age:")
display(
    data.groupby("Age")["Selling_Price"]
    .mean()
    .sort_index()
    .round(2)
    .to_frame("Average Selling Price")
)

print("Conclusion: Vehicle age has a clear negative relationship with selling price.")
print("Kilometres driven has a weaker direct relationship with selling price in this dataset.")


## 14. Newest Vehicles (Manufactured After 2014)

In [ ]:
newest = data[data["Year"] > 2014]

print("Records manufactured after 2014:", len(newest))
print("Unique vehicle models manufactured after 2014:", newest["Car_Name"].nunique())

display(newest.sort_values("Year", ascending=False).head(20))


## 15. Identify Two-Wheelers

In [ ]:
# This dataset has no explicit vehicle-category column.
# Two-wheelers are classified using common two-wheeler manufacturers/models.

two_wheeler_brands = [
    "Bajaj", "Hero", "Honda", "Yamaha", "TVS",
    "Royal", "KTM", "Activa", "Suzuki", "Mahindra"
]

# Use model-name keywords to identify two-wheelers more safely
two_wheeler_keywords = [
    "Pulsar", "Activa", "Splendor", "Discover", "Apache",
    "FZ", "FZS", "Classic", "Thunderbird", "Bullet",
    "Avenger", "Platina", "Duke", "R15", "Vario",
    "CB", "Gixxer", "V15", "Karizma", "Star City",
    "Scooty", "Wego", "Maestro", "Passion", "Glamour",
    "Stunner", "Shine", "Dream", "HF Deluxe", "CT 100",
    "Victor", "Street 750"
]

pattern = "|".join(two_wheeler_keywords)
data["Is_Two_Wheeler"] = data["Car_Name"].str.contains(
    pattern, case=False, regex=True, na=False
)

two_wheelers = data[data["Is_Two_Wheeler"]].copy()
cars = data[~data["Is_Two_Wheeler"]].copy()

print("Two-wheeler records:", len(two_wheelers))
print("Car records:", len(cars))


## 16. Two-Wheeler Analysis

In [ ]:
print("Oldest two-wheeler:")
oldest_bike = two_wheelers.loc[two_wheelers["Year"].idxmin()]
display(oldest_bike.to_frame("Value"))

print("\nNewest two-wheeler manufacturing year:", two_wheelers["Year"].max())

print("\nMost frequently occurring two-wheelers:")
display(two_wheelers["Car_Name"].value_counts().head(10).to_frame("Records"))

bike_exception = two_wheelers.loc[
    two_wheelers["Depreciation_Percent"].idxmin()
]

print("\nTwo-wheeler with lowest percentage depreciation:")
display(bike_exception.to_frame("Value"))


## 17. Car-Only Analysis

In [ ]:
print("Number of car records:", len(cars))
print("Unique car models:", cars["Car_Name"].nunique())

print("\nOldest car(s):")
oldest_car_year = cars["Year"].min()
display(cars[cars["Year"] == oldest_car_year].sort_values("Car_Name"))

print("\nNewest car(s):")
newest_car_year = cars["Year"].max()
display(cars[cars["Year"] == newest_car_year].sort_values("Car_Name"))

print("\nMost frequently occurring cars:")
display(cars["Car_Name"].value_counts().head(10).to_frame("Records"))


## 18. Exceptional Resale Deals

In [ ]:
# Lowest depreciation percentage among vehicles with reasonable data
exceptional = data.sort_values("Depreciation_Percent").head(10)

display(
    exceptional[
        [
            "Car_Name", "Year", "Present_Price", "Selling_Price",
            "Depreciation", "Depreciation_Percent",
            "Kms_Driven", "Owner", "Seller_Type", "Fuel_Type", "Transmission"
        ]
    ].round(2)
)

print("These vehicles have unusually high resale-value retention.")


## 19. Visualizations

In [ ]:
# Manufacturing year distribution
data["Year"].value_counts().sort_index().plot(kind="bar")
plt.title("Vehicles by Manufacturing Year")
plt.xlabel("Manufacturing Year")
plt.ylabel("Number of Vehicles")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Selling price distribution
data["Selling_Price"].plot(kind="hist", bins=30)
plt.title("Selling Price Distribution")
plt.xlabel("Selling Price (lakh)")
plt.ylabel("Number of Vehicles")
plt.tight_layout()
plt.show()


In [ ]:
# Age vs selling price
plt.scatter(data["Age"], data["Selling_Price"], alpha=0.6)
plt.title("Vehicle Age vs Selling Price")
plt.xlabel("Vehicle Age")
plt.ylabel("Selling Price (lakh)")
plt.tight_layout()
plt.show()


In [ ]:
# Kms driven vs selling price
plt.scatter(data["Kms_Driven"], data["Selling_Price"], alpha=0.6)
plt.title("Kilometres Driven vs Selling Price")
plt.xlabel("Kilometres Driven")
plt.ylabel("Selling Price (lakh)")
plt.tight_layout()
plt.show()


## 20. Final Answers to the 25 Questions

In [ ]:
answers = {
    1: f"Manufacturing years range from {data['Year'].min()} to {data['Year'].max()}.",
    2: f"Lowest selling price = {data['Selling_Price'].min():.2f} lakh.",
    3: f"Highest selling price = {data['Selling_Price'].max():.2f} lakh.",
    4: f"There are {len(data)} records.",
    5: "There are no missing values." if data.isnull().sum().sum() == 0 else "Missing values are present.",
    6: f"There are {data['Car_Name'].nunique()} unique vehicle models.",
    7: f"Most frequently occurring vehicle = {data['Car_Name'].value_counts().idxmax()} ({data['Car_Name'].value_counts().max()} records).",
    8: f"Yes, there are {(data['Fuel_Type'] == 'CNG').sum()} CNG vehicles.",
    9: f"There are {(data['Seller_Type'] == 'Individual').sum()} vehicles sold by individuals.",
    10: f"Yes, there are {(data['Transmission'] == 'Automatic').sum()} automatic vehicles.",
    11: f"There are {(data['Owner'] == 0).sum()} first-owner vehicles (Owner = 0).",
    12: f"Highest absolute depreciation = {most_depreciated['Car_Name']}; lowest = {least_depreciated['Car_Name']}.",
    13: "Brands with lower average percentage depreciation generally retain value better; use the brand table above.",
    14: "Vehicle age is the strongest observable factor affecting depreciation, followed by usage and vehicle/model characteristics.",
    15: "Selling price generally decreases with vehicle age; kilometres driven has a weaker direct relationship with selling price.",
    16: f"There are {len(newest)} records manufactured after 2014.",
    17: f"There are {len(two_wheelers)} two-wheeler records under the classification used above.",
    18: f"Oldest two-wheeler = {oldest_bike['Car_Name']} ({int(oldest_bike['Year'])}).",
    19: f"Newest two-wheeler manufacturing year = {int(two_wheelers['Year'].max())}.",
    20: f"Most frequently occurring two-wheeler = {two_wheelers['Car_Name'].value_counts().idxmax()}.",
    21: f"An exceptional two-wheeler resale case is {bike_exception['Car_Name']} with approximately {bike_exception['Depreciation_Percent']:.2f}% depreciation.",
    22: f"Yes, there are {len(cars)} car records under the classification used above.",
    23: f"Oldest car manufacturing year = {int(oldest_car_year)}.",
    24: f"Newest car manufacturing year = {int(newest_car_year)}.",
    25: f"An exceptional car resale case can be found among the lowest-depreciation cars; see the table above."
}

for q, answer in answers.items():
    print(f"{q}. {answer}")


## Conclusion

The analysis shows that **vehicle age is the clearest factor associated with depreciation**. Present price is strongly related to selling price, while kilometres driven has a weaker direct relationship with selling price but a more noticeable relationship with depreciation.

The dataset is clean with no missing values, contains both cars and two-wheelers, and includes petrol, diesel and a small number of CNG vehicles. A few newer/low-usage vehicles retain a particularly high proportion of their original value.
